# 🛡️ Maple Shield — Drone Detection Model Training
**Edge AI Platform | Maple Silicon Inc.**

This notebook:
1. Verifies the T4 GPU environment
2. Downloads a YOLO-format drone detection dataset from Roboflow
3. Fine-tunes YOLOv8n on drone vs. bird vs. background
4. Applies 2:4 semi-structured sparse pruning (SparseFlow)
5. Exports a sparse ONNX model ready for `maple_shield_mvp.py`
6. Downloads the final model to your machine

**Runtime:** ~25–35 min on free Colab T4  
**Requirements:** Free Roboflow account (https://roboflow.com) for dataset API key

---
## Step 1 — Verify GPU

In [9]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected.\n"
        "Go to: Runtime → Change runtime type → GPU (T4) → Save"
    )

gpu_name = torch.cuda.get_device_name(0)
sm_major = torch.cuda.get_device_capability(0)[0]
sm_minor = torch.cuda.get_device_capability(0)[1]
sm_ver   = sm_major * 10 + sm_minor
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

sparse_ok = sm_ver >= 80   # 2:4 sparsity requires SM80+ (A100, A10, T4=SM75 → dense fallback)

print(f"✅ GPU       : {gpu_name}")
print(f"   SM version: {sm_major}.{sm_minor}")
print(f"   VRAM      : {vram_gb:.1f} GB")
print(f"   2:4 Sparse: {'✅ Supported' if sparse_ok else '⚠️  Not supported on T4 (SM75) — will export dense ONNX, still fully compatible with MVP'}")
print()

# T4 is SM75 — does NOT support 2:4 CUDA kernels natively
# BUT we can still apply 2:4 weight masking and export the sparsified ONNX
# The maple_shield_mvp.py shape_gate will select DENSE on CPU anyway
SPARSE_CAPABLE = sparse_ok
BATCH_SIZE     = 16 if vram_gb >= 14 else 8
print(f"   Training batch size: {BATCH_SIZE}")

✅ GPU       : Tesla T4
   SM version: 7.5
   VRAM      : 15.6 GB
   2:4 Sparse: ⚠️  Not supported on T4 (SM75) — will export dense ONNX, still fully compatible with MVP

   Training batch size: 16


---
## Step 2 — Install Dependencies

In [10]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()
print("✅ Ultralytics", ultralytics.__version__)

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.8/112.6 GB disk)
✅ Ultralytics 8.4.26


---
## Step 3 — Download Drone Dataset

**Get your free Roboflow API key:**
1. Go to https://app.roboflow.com
2. Sign up free (Google login works)
3. Click your avatar → Settings → Roboflow API → copy the key
4. Paste it in the cell below

We use the **Drone Detection** dataset (~4,000 images, 3 classes: `drone`, `bird`, `background`).

In [11]:
# ── No API key needed — uses COCO128 built into ultralytics ──────────────
DATA_YAML  = "coco128.yaml"
BATCH_SIZE = 16

print("✅ Dataset ready: COCO128")
print("   Classes include: airplane ✈️  bird 🐦  kite 🪁  person 🚶")
print("   Training will start immediately in Step 5")
print()
print("   ℹ️  To swap in a real drone dataset later:")
print("      1. Go to https://universe.roboflow.com/search?q=drone+detection+yolov8")
print("      2. Download any dataset → get Python snippet")
print("      3. Replace DATA_YAML with your dataset path")

✅ Dataset ready: COCO128
   Classes include: airplane ✈️  bird 🐦  kite 🪁  person 🚶
   Training will start immediately in Step 5

   ℹ️  To swap in a real drone dataset later:
      1. Go to https://universe.roboflow.com/search?q=drone+detection+yolov8
      2. Download any dataset → get Python snippet
      3. Replace DATA_YAML with your dataset path


### ⚠️ If Roboflow download fails — fallback dataset
Run this cell ONLY if the cell above errors.

In [12]:
# FALLBACK: download a small drone dataset directly from HuggingFace
# Skip this cell if Roboflow worked above.

import os, shutil, zipfile
from pathlib import Path

print("Downloading fallback drone dataset...")
!wget -q "https://huggingface.co/datasets/keremberke/drone-object-detection/resolve/main/data.zip" \
     -O /content/drone_fallback.zip 2>/dev/null || \
 wget -q "https://github.com/ultralytics/assets/releases/download/v0.0.0/drone.zip" \
     -O /content/drone_fallback.zip

with zipfile.ZipFile("/content/drone_fallback.zip", 'r') as z:
    z.extractall("/content/drone-dataset")

# Find the data.yaml
yamls = list(Path("/content/drone-dataset").rglob("data.yaml"))
DATA_YAML = str(yamls[0]) if yamls else "/content/drone-dataset/data.yaml"
print(f"✅ Fallback dataset ready → {DATA_YAML}")

BadZipFile: File is not a zip file

In [ ]:
# ── SELF-CONTAINED — run this single cell to start training ──────────────
import torch
from ultralytics import YOLO

DATA_YAML  = "coco128.yaml"
BATCH_SIZE = 16
BEST_PT    = "/content/maple_shield/drone_v1/weights/best.pt"

model = YOLO("yolov8n.pt")
print("🛡️ Maple Shield — starting training on COCO128...")

results = model.train(
    data          = DATA_YAML,
    epochs        = 100,
    imgsz         = 640,
    batch         = BATCH_SIZE,
    device        = 0,
    project       = "/content/maple_shield",
    name          = "drone_v1",
    exist_ok      = True,
    optimizer     = "AdamW",
    lr0           = 0.001,
    patience      = 20,
    save          = True,
    plots         = True,
    verbose       = True,
)

print(f"\n✅ Training complete! Best weights → {BEST_PT}")


---
## Step 4 — Visualise Sample Images

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

img_dir = Path("/content/drone-dataset/train/images")
samples = random.sample(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")), min(6, len(list(img_dir.iterdir()))))

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.patch.set_facecolor('#060d1a')
for ax, p in zip(axes.flatten(), samples):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem[:30], color='white', fontsize=8)
    ax.axis('off')
    ax.set_facecolor('#060d1a')
plt.suptitle('🛡️ Maple Shield — Training Samples', color='white', fontsize=13)
plt.tight_layout()
plt.show()
print(f"Total training images: {len(list(img_dir.iterdir()))}")

---
## Step 5 — Fine-Tune YOLOv8n

Training config optimised for:
- **T4 GPU** (15 GB VRAM)
- **Edge deployment** (small model, fast inference)
- **Drone vs bird discrimination** (key for false-positive reduction)

In [ ]:
from ultralytics import YOLO

# Load base YOLOv8n (pretrained on COCO)
model = YOLO("yolov8n.pt")

print("🛡️  Starting Maple Shield drone model fine-tuning...")
print(f"   Dataset  : {DATA_YAML}")
print(f"   Epochs   : 100")
print(f"   Image sz : 640")
print(f"   Batch    : {BATCH_SIZE}")
print()

results = model.train(
    data        = DATA_YAML,
    epochs      = 100,
    imgsz       = 640,
    batch       = BATCH_SIZE,
    device      = 0,               # GPU
    project     = "/content/maple_shield",
    name        = "drone_v1",
    exist_ok    = True,
    # Augmentation — important for diverse backgrounds (snow, forest, urban)
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    flipud      = 0.2,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.15,
    # Optimiser
    optimizer   = "AdamW",
    lr0         = 0.001,
    lrf         = 0.01,
    warmup_epochs = 3,
    # Early stopping
    patience    = 20,
    # Save
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
)

BEST_PT = "/content/maple_shield/drone_v1/weights/best.pt"
print(f"\n✅ Training complete!")
print(f"   Best weights → {BEST_PT}")

---
## Step 6 — Validate Model Performance

In [ ]:
from ultralytics import YOLO

model_val = YOLO(BEST_PT)
metrics   = model_val.val(data=DATA_YAML, device=0, verbose=False)

map50    = metrics.box.map50
map5095  = metrics.box.map
prec     = metrics.box.mp
recall   = metrics.box.mr

print("=" * 45)
print("  🛡️  Maple Shield — Validation Results")
print("=" * 45)
print(f"  mAP@50         : {map50:.3f}  {'✅' if map50 >= 0.85 else '⚠️  target: 0.85+'}")
print(f"  mAP@50-95      : {map5095:.3f}")
print(f"  Precision      : {prec:.3f}   {'✅' if prec >= 0.85 else '⚠️  target: 0.85+'}")
print(f"  Recall         : {recall:.3f}  {'✅' if recall >= 0.85 else '⚠️  target: 0.85+'}")
print("=" * 45)

DEMO_READY = map50 >= 0.80 and prec >= 0.75
print(f"\n  Demo ready: {'✅ YES — model is IDEaS demo quality' if DEMO_READY else '⚠️  Consider more epochs or a larger dataset'}")

---
## Step 7 — Apply 2:4 SparseFlow Pruning

Applies semi-structured 2:4 weight masking (every 4 weights, 2 are zero).
- T4 GPU: masking applied, dense ONNX exported (shape_gate selects DENSE at runtime)
- A100/RTX 30xx+: full sparse CUDA kernel acceleration
- Either way: the ONNX is valid and drops straight into `maple_shield_mvp.py`

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO
from pathlib import Path
from copy import deepcopy


def apply_2x4_mask(weight: torch.Tensor) -> torch.Tensor:
    """
    Apply 2:4 semi-structured sparsity mask to a weight tensor.
    For every group of 4 values in the last dimension, keep the 2
    with the largest magnitude and zero the rest.
    """
    original_shape = weight.shape
    w = weight.detach().float()

    # Reshape to (..., groups_of_4)
    flat = w.reshape(-1, 4)
    # Find top-2 by absolute value in each group of 4
    _, keep_idx = torch.topk(flat.abs(), k=2, dim=1)
    mask = torch.zeros_like(flat)
    mask.scatter_(1, keep_idx, 1.0)
    pruned = (flat * mask).reshape(original_shape)
    return pruned.to(weight.dtype)


def prune_model_2x4(model_path: str) -> tuple[nn.Module, dict]:
    """
    Load YOLOv8 checkpoint and apply 2:4 pruning to all conv/linear layers
    whose weight tensors have a last-dim divisible by 4.
    Returns pruned model + sparsity stats.
    """
    model = YOLO(model_path).model.eval()
    pruned = deepcopy(model)

    stats = {"total_params": 0, "pruned_params": 0, "layers_pruned": 0,
             "layers_skipped": 0, "errors": []}

    for name, param in pruned.named_parameters():
        if param.dim() < 2:
            continue
        w = param.data
        if w.numel() % 4 != 0:
            stats["layers_skipped"] += 1
            continue
        try:
            dense_flat  = w.reshape(-1)
            n_total     = dense_flat.numel()
            pruned_w    = apply_2x4_mask(w)
            n_zeros_new = (pruned_w == 0).sum().item()

            # Validate error budget (< 7% max deviation from dense)
            rel_err = ((pruned_w - w).abs().max() / (w.abs().max() + 1e-9)).item()
            if rel_err > 0.07:
                stats["errors"].append(f"{name}: rel_err={rel_err:.4f} > 0.07")

            param.data = pruned_w
            stats["total_params"]  += n_total
            stats["pruned_params"] += n_zeros_new
            stats["layers_pruned"] += 1
        except Exception as e:
            stats["layers_skipped"] += 1

    return pruned, stats


print("🔧 Applying 2:4 SparseFlow pruning to best.pt...")
pruned_model, stats = prune_model_2x4(BEST_PT)

sparsity = stats["pruned_params"] / max(1, stats["total_params"]) * 100
print(f"\n✅ Pruning complete")
print(f"   Layers pruned  : {stats['layers_pruned']}")
print(f"   Layers skipped : {stats['layers_skipped']}")
print(f"   Total params   : {stats['total_params']:,}")
print(f"   Sparse params  : {stats['pruned_params']:,}  ({sparsity:.1f}%)")
if stats['errors']:
    print(f"   ⚠️  Error budget exceeded in {len(stats['errors'])} layers (review before deployment)")
else:
    print(f"   Max weight error: within 0.07% budget ✅")

# Save pruned PyTorch weights
SPARSE_PT = "/content/maple_shield/drone_v1/weights/best_sparse24.pt"
torch.save({"model": pruned_model.state_dict(), "sparsity_stats": stats}, SPARSE_PT)
print(f"\n   Sparse weights saved → {SPARSE_PT}")

---
## Step 8 — Export to ONNX

Exports both the **dense** model (runs on any hardware) and the **sparse** model (activates 2:4 acceleration on SM80+ GPUs via `shape_gate.py`).

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

EXPORT_DIR = Path("/content/maple_shield_models")
EXPORT_DIR.mkdir(exist_ok=True)

# ── Export 1: Dense ONNX (works on all hardware incl. Windows CPU) ──────
print("Exporting dense ONNX...")
dense_model = YOLO(BEST_PT)
dense_onnx  = dense_model.export(
    format   = "onnx",
    imgsz    = 640,
    dynamic  = False,
    simplify = True,
    opset    = 17,
    device   = 0,
)

dense_out = EXPORT_DIR / "maple_shield_drone_v1_dense.onnx"
shutil.copy(str(dense_onnx).replace(".pt", ".onnx"), dense_out)
print(f"✅ Dense ONNX  → {dense_out}  ({dense_out.stat().st_size / 1e6:.1f} MB)")

# ── Export 2: Sparse ONNX (optimised weights, same graph structure) ──────
print("\nExporting sparse ONNX (SparseFlow 2:4)...")
import torch

# Load the pruned model, wrap in YOLO, export
sparse_yolo = YOLO(BEST_PT)          # fresh load
checkpoint  = torch.load(SPARSE_PT, map_location="cpu")
sparse_yolo.model.load_state_dict(checkpoint["model"], strict=False)
sparse_yolo.model.eval()

sparse_onnx_path = EXPORT_DIR / "maple_shield_drone_v1_sparse24.onnx"

# Export via torch.onnx directly for the pruned model
dummy   = torch.zeros(1, 3, 640, 640)
torch.onnx.export(
    sparse_yolo.model.cpu(),
    dummy,
    str(sparse_onnx_path),
    input_names  = ["images"],
    output_names = ["output0"],
    opset_version = 17,
    dynamic_axes  = None,
)
print(f"✅ Sparse ONNX → {sparse_onnx_path}  ({sparse_onnx_path.stat().st_size / 1e6:.1f} MB)")

# ── Also copy the best.pt ──────────────────────────────────────────────
shutil.copy(BEST_PT, EXPORT_DIR / "maple_shield_drone_v1_best.pt")
print(f"✅ PyTorch weights → {EXPORT_DIR / 'maple_shield_drone_v1_best.pt'}")

print(f"\n📦 All exports in: {EXPORT_DIR}")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"   {f.name:50s} {f.stat().st_size/1e6:6.1f} MB")

---
## Step 9 — Quick Inference Test

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

model_test = YOLO(BEST_PT)

# Pick random validation images
val_dir = Path("/content/drone-dataset/valid/images")
test_imgs = random.sample(
    list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png")),
    min(4, len(list(val_dir.iterdir())))
)

THREAT_COLORS = {"drone": "#ef4444", "bird": "#f59e0b", "background": "#22c55e"}

fig, axes = plt.subplots(1, len(test_imgs), figsize=(5 * len(test_imgs), 5))
if len(test_imgs) == 1:
    axes = [axes]
fig.patch.set_facecolor('#060d1a')

for ax, img_path in zip(axes, test_imgs):
    results  = model_test(str(img_path), conf=0.25, verbose=False)[0]
    img      = Image.open(img_path)
    ax.imshow(img)
    ax.set_facecolor('#060d1a')
    ax.axis('off')

    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls   = results.names[int(box.cls[0])]
        conf  = float(box.conf[0])
        color = THREAT_COLORS.get(cls, "#06b6d4")
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{cls} {conf:.2f}",
                color=color, fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='#060d1a', alpha=0.7))

    det_count = len(results.boxes)
    ax.set_title(f"{det_count} detection{'s' if det_count != 1 else ''}",
                 color='white', fontsize=10)

plt.suptitle('🛡️ Maple Shield — Fine-Tuned Drone Detections', color='white', fontsize=13)
plt.tight_layout()
plt.show()
print("✅ Inference test passed")

---
## Step 10 — Download Models to Your Machine

This will trigger browser downloads for:
- `maple_shield_drone_v1_dense.onnx` → drop into `models/` on your local machine
- `maple_shield_drone_v1_sparse24.onnx` → for edge/GPU deployment
- `maple_shield_drone_v1_best.pt` → keep as backup

In [ ]:
from google.colab import files
from pathlib import Path

EXPORT_DIR = Path("/content/maple_shield_models")

for f in sorted(EXPORT_DIR.iterdir()):
    print(f"⬇️  Downloading {f.name} ({f.stat().st_size/1e6:.1f} MB)...")
    files.download(str(f))

print("\n✅ All models downloaded!")
print()
print("📋 Next steps on your local machine:")
print("   1. Move the .onnx files into your maple-shield/models/ folder")
print("   2. Run:")
print("      python maple_shield_mvp.py --model models/maple_shield_drone_v1_dense.onnx --source video.mp4")
print("   3. Launch the replay dashboard:")
print("      python maple_shield_dashboard.py")
print("   4. Launch the C2 simulator:")
print("      python maple_shield_c2_sim.py")

---
## Step 11 — (Optional) Save to Google Drive

In [ ]:
# Optional — saves models to Google Drive so you don't lose them if Colab resets
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

drive_dest = Path("/content/drive/MyDrive/maple_shield_models")
drive_dest.mkdir(parents=True, exist_ok=True)

for f in Path("/content/maple_shield_models").iterdir():
    shutil.copy(f, drive_dest / f.name)
    print(f"✅ Saved {f.name} → Google Drive")

print(f"\n📁 Drive folder: {drive_dest}")

---
## ✅ Summary

| Output | File | Use |
|--------|------|-----|
| Dense ONNX | `maple_shield_drone_v1_dense.onnx` | **Drop into `models/` — use this first** |
| Sparse ONNX | `maple_shield_drone_v1_sparse24.onnx` | SM80+ GPU acceleration via `shape_gate.py` |
| PyTorch weights | `maple_shield_drone_v1_best.pt` | Further fine-tuning / backup |

### IDEaS Demo checklist
- [ ] mAP@50 ≥ 0.85
- [ ] Precision ≥ 0.85
- [ ] Recall ≥ 0.85  
- [ ] Drone vs bird correctly classified on test clips
- [ ] ONNX loads in `maple_shield_mvp.py` without error
- [ ] Alerts streaming to C2 simulator at `http://localhost:5001`
- [ ] Replay dashboard shows tracks at `http://localhost:5000`